# Submit the Customer H2O Scoring Pipeline

This notebook generates and runs the customer native-binary or MOJO offline scoring pipeline using the environment prepared in notebook 03. The batch scorer, component YAML, and pipeline YAML are written under ignored `outputs/generated` before submission.

The generated pipeline is printed and can be handed to Airflow with the same immutable model, environment, data, compute, and correlation identifiers.

## Before you run it

- If golden validation is required, confirm notebook 04 passed parity for this model and environment version.
- Confirm `.env` still describes the bundle validated in notebook 01. Replacement steps are in `data/h2o/customer_bundle/README.md`.
- Confirm the scoring input CSV passed the intake checks.
- Review the compute, output datastore, correlation ID, ID column, and reject policy in `workshop/.env`.
- Leave `RUN_H2O_SCORING_PIPELINE=false` for the review pass.

We will prepare the input asset, bind the YAML pipeline, review the resolved run settings, and submit only when the safety switch is enabled.

**Source:** Adapted from this repository's H2O scoring pipeline and notebook submission patterns.

In [11]:
from pathlib import Path
import json
import os
import sys

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data, ManagedIdentityConfiguration
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def enabled(name: str) -> bool:
    return os.getenv(name, "false").lower() in {"1", "true", "yes"}

sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from customer_profiles import load_selected_manifest

print(f"Workshop root: {WORKSHOP_ROOT}")

Workshop root: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop


## 1. Review the run contract

These values identify one repeatable scoring run. The model, environment, and input are versioned Azure ML assets. The correlation ID ties the run to an upstream request, while the ID column and reject policy control record-level handling.

Check the summary before connecting to Azure. If the local input path is wrong, the next cell stops immediately.

In [ ]:
BUNDLE_DIR, manifest = load_selected_manifest(WORKSHOP_ROOT)
MODEL_PATH = BUNDLE_DIR / manifest["model_file"]
golden_data = manifest.get("golden_data", {})

input_value = os.getenv(
    "H2O_CUSTOMER_SCORING_INPUT_PATH",
    "",
).strip()
if input_value:
    input_path_value = Path(input_value)
    INPUT_PATH = (
        input_path_value
        if input_path_value.is_absolute()
        else WORKSHOP_ROOT / input_path_value
    ).resolve()
elif golden_data.get("provided", False):
    INPUT_PATH = (BUNDLE_DIR / golden_data["input_file"]).resolve()
else:
    raise ValueError(
        "Set H2O_CUSTOMER_SCORING_INPUT_PATH because the selected profile "
        "does not contain golden input data"
    )
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Customer scoring input not found: {INPUT_PATH}")

GOLDEN_VALIDATION_REQUIRED = golden_data.get("required", False)
RUNTIME_VALIDATION_PATH = (
    WORKSHOP_ROOT
    / "outputs/generated/h2o_customer/online/runtime_validation.json"
)

MODEL_NAME = manifest["model_name"]
ENVIRONMENT_NAME = manifest["environment_name"]
DATA_NAME = manifest["input_data_name"]
EXPERIMENT_NAME = manifest["experiment_name"]
os.environ["H2O_CUSTOMER_EXPERIMENT_NAME"] = EXPERIMENT_NAME
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
COMPUTE_IDENTITY_CLIENT_ID = os.environ[
    "AZUREML_COMPUTE_IDENTITY_CLIENT_ID"
].strip()
if not COMPUTE_IDENTITY_CLIENT_ID:
    raise ValueError(
        "AZUREML_COMPUTE_IDENTITY_CLIENT_ID must identify the compute cluster UMI"
    )
OUTPUT_DATASTORE = os.getenv(
    "AZUREML_OUTPUT_DATASTORE",
    "workspaceblobstore",
)
CORRELATION_ID = os.getenv(
    "H2O_CORRELATION_ID",
    "workshop-manual-run",
)
ID_COLUMN = os.getenv("H2O_ID_COLUMN", "__generated__")
FAIL_ON_REJECTS = enabled("H2O_FAIL_ON_REJECTS")
RUN = enabled("RUN_H2O_SCORING_PIPELINE")

display(
    {
        "profile": manifest["profile"],
        "model": f"{MODEL_NAME}@latest",
        "environment": f"{ENVIRONMENT_NAME}@latest",
        "input_asset": f"{DATA_NAME} (version assigned on submission)",
        "input_file": str(INPUT_PATH),
        "input_source": "configured" if input_value else "selected profile golden input",
        "compute": COMPUTE_NAME,
        "output_datastore": OUTPUT_DATASTORE,
        "runtime_identity": COMPUTE_IDENTITY_CLIENT_ID,
        "experiment": EXPERIMENT_NAME,
        "correlation_id": CORRELATION_ID,
        "id_column": ID_COLUMN,
        "fail_on_rejects": FAIL_ON_REJECTS,
        "golden_validation_required": GOLDEN_VALIDATION_REQUIRED,
        "runtime_validation_evidence": str(RUNTIME_VALIDATION_PATH),
    }
)

## 2. Confirm the target workspace and compute

The next cell is read-only. It checks that the workspace and compute named in `.env` exist and prints the submission switch.

For the review pass, `Submission enabled` should be `False`.

In [13]:
credential = AzureCliCredential(
    tenant_id=os.getenv("AZURE_TENANT_ID") or None
)
ml_client = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)

workspace = ml_client.workspaces.get(os.environ["AZUREML_WORKSPACE_NAME"])
compute = ml_client.compute.get(COMPUTE_NAME)

print(f"Target workspace: {workspace.name}")
print(f"Compute: {compute.name}")
print(f"Submission enabled: {RUN}")

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Target workspace: mlwdevcc01
Compute: aml-cluster-dev-cc01
Submission enabled: True


## 3. Prepare the input asset

Azure ML jobs consume a registered URI file rather than a path on this compute instance. We define the customer CSV as an immutable data asset here, but registration is deferred to the guarded submission cell.

When submission is enabled, Azure ML assigns the next asset version and the submitted job is bound to that concrete version.

In [14]:
data_definition = Data(
    name=DATA_NAME,
    type=AssetTypes.URI_FILE,
    path=str(INPUT_PATH),
    description="Customer H2O scoring input",
    tags={
        "workshop": "azureml-h2o-customer",
        "purpose": "offline-scoring",
    },
)

print(f"Input asset: {data_definition.name} (version assigned on registration)")
print(f"Type: {data_definition.type}")
print(f"Local path: {data_definition.path}")

Input asset: workshop-h2o-customer-binary-input (version assigned on registration)
Type: uri_file
Local path: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/h2o_customer_bundle/golden_input.csv


## 4. Generate the batch scorer and pipeline definitions

This notebook owns its H2O batch artifacts. It writes the dual-format scorer, command-component YAML, and pipeline YAML under ignored `outputs/generated`, then validates them before loading the job.

In [ ]:
import ast
import textwrap
import yaml

GENERATED_PIPELINE_DIR = WORKSHOP_ROOT / "outputs/generated/h2o_customer/batch"
GENERATED_CODE_DIR = GENERATED_PIPELINE_DIR / "code"
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)
(GENERATED_CODE_DIR / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
BATCH_SCORE_PATH = GENERATED_CODE_DIR / "score.py"
COMPONENT_PATH = GENERATED_PIPELINE_DIR / "h2o-score.yaml"
PIPELINE_PATH = GENERATED_PIPELINE_DIR / "pipeline.yaml"

BATCH_SCORE_SOURCE = r'''
from __future__ import annotations

import argparse
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import h2o
import mlflow
import numpy as np
import pandas as pd


def parse_bool(value):
    normalized = str(value).strip().lower()
    if normalized in {"1", "true", "yes", "y"}:
        return True
    if normalized in {"0", "false", "no", "n"}:
        return False
    raise argparse.ArgumentTypeError(f"Expected a boolean, received {value!r}")


def parse_args():
    parser = argparse.ArgumentParser(description="Score a CSV with an H2O model")
    parser.add_argument("--model-dir", required=True)
    parser.add_argument("--input-data", required=True)
    parser.add_argument("--scored-output", required=True)
    parser.add_argument("--monitoring-output", required=True)
    parser.add_argument("--correlation-id", required=True)
    parser.add_argument("--id-column", default="__generated__")
    parser.add_argument("--fail-on-rejects", type=parse_bool, default=False)
    parser.add_argument("--h2o-nthreads", type=int, default=3)
    parser.add_argument("--h2o-max-mem-size", default="6G")
    return parser.parse_args()


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_one(root, name):
    matches = list(Path(root).rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {name}, found {len(matches)}")
    return matches[0]


def resolve_csv(path):
    path = Path(path)
    if path.is_file():
        return path
    matches = sorted(path.rglob("*.csv"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one input CSV, found {len(matches)}")
    return matches[0]


def prepare_input(path, manifest, id_column):
    source_path = resolve_csv(path)
    source = pd.read_csv(source_path)
    if source.empty:
        raise ValueError("Input CSV contains no rows")
    features = manifest["features"]
    missing = [column for column in features if column not in source.columns]
    if missing:
        raise ValueError(f"Missing required feature columns: {missing}")
    if id_column and id_column != "__generated__":
        if id_column not in source.columns:
            raise ValueError(f"Configured ID column is missing: {id_column}")
        row_ids = source[id_column].astype("string")
        if row_ids.isna().any() or row_ids.duplicated().any():
            raise ValueError("Configured ID values must be non-null and unique")
    else:
        width = max(6, len(str(len(source))))
        row_ids = pd.Series(
            [f"{source_path.stem}:{index:0{width}d}" for index in range(len(source))],
            index=source.index,
            dtype="string",
        )
    prepared = pd.DataFrame(index=source.index)
    categorical = set(manifest.get("categorical_features", []))
    invalid_masks = {}
    invalid_counts = {}
    for feature in features:
        if feature in categorical:
            prepared[feature] = source[feature].astype("string")
            invalid_masks[feature] = prepared[feature].isna()
        else:
            prepared[feature] = pd.to_numeric(source[feature], errors="coerce")
            invalid_masks[feature] = ~np.isfinite(prepared[feature])
        invalid_counts[feature] = int(invalid_masks[feature].sum())
    invalid = pd.DataFrame(invalid_masks).any(axis=1)
    reasons = []
    for index in prepared.index[invalid]:
        names = [feature for feature in features if invalid_masks[feature].at[index]]
        reasons.append("invalid_or_null:" + ",".join(names))
    rejects = pd.DataFrame(
        {
            "row_id": row_ids.loc[invalid].astype(str).to_numpy(),
            "source_file": source_path.name,
            "status": "rejected",
            "rejection_reason": reasons,
        }
    )
    valid = ~invalid
    return (
        prepared.loc[valid, features].reset_index(drop=True),
        row_ids.loc[valid].reset_index(drop=True),
        rejects,
        invalid_counts,
        source_path.name,
    )


def main():
    args = parse_args()
    started = time.perf_counter()
    scoring_time = datetime.now(timezone.utc).isoformat()
    aml_run_id = os.getenv("AZUREML_RUN_ID", "local")
    scored_output = Path(args.scored_output)
    monitoring_output = Path(args.monitoring_output)
    scored_output.mkdir(parents=True, exist_ok=True)
    monitoring_output.mkdir(parents=True, exist_ok=True)
    manifest_path = find_one(Path(args.model_dir).resolve(), "model_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    model_path = manifest_path.parent / manifest["model_file"]
    model_format = manifest.get("model_format")
    if model_format not in {"h2o_binary", "h2o_mojo"}:
        raise RuntimeError(f"Unsupported H2O model format: {model_format}")
    runtime_version = manifest.get("runtime_h2o_version", manifest["h2o_version"])
    if h2o.__version__ != runtime_version:
        raise RuntimeError(f"Expected runtime h2o=={runtime_version}, found {h2o.__version__}")
    if model_format == "h2o_binary" and runtime_version != manifest["h2o_version"]:
        raise RuntimeError("Native H2O binaries require the producer version")
    model_sha = sha256(model_path)
    if model_sha != manifest["files"].get(model_path.name):
        raise RuntimeError("Model checksum does not match the manifest")
    valid, valid_ids, rejects, invalid_counts, source_file = prepare_input(
        args.input_data, manifest, args.id_column
    )
    predictions = pd.Series(dtype="object")
    jvm_seconds = 0.0
    scoring_seconds = 0.0
    try:
        if len(valid):
            h2o.no_progress()
            jvm_started = time.perf_counter()
            h2o.init(
                ip="127.0.0.1",
                port=54321,
                start_h2o=True,
                nthreads=args.h2o_nthreads,
                max_mem_size=args.h2o_max_mem_size,
                strict_version_check=True,
                bind_to_localhost=True,
                verbose=False,
                telemetry=False,
            )
            model = (
                h2o.load_model(str(model_path))
                if model_format == "h2o_binary"
                else h2o.upload_mojo(str(model_path))
            )
            jvm_seconds = time.perf_counter() - jvm_started
            scoring_started = time.perf_counter()
            h2o_frame = h2o.H2OFrame(valid)
            for column in manifest.get("categorical_features", []):
                h2o_frame[column] = h2o_frame[column].asfactor()
            prediction_frame = model.predict(h2o_frame)
            predictions = prediction_frame.as_data_frame()["predict"].reset_index(drop=True)
            scoring_seconds = time.perf_counter() - scoring_started
            h2o.remove(prediction_frame)
            h2o.remove(h2o_frame)
    finally:
        if h2o.connection() is not None:
            h2o.cluster().shutdown(prompt=False)
    common = {
        "model_name": manifest["model_name"],
        "model_version": manifest["model_version"],
        "model_format": model_format,
        "model_sha256": model_sha,
        "model_h2o_version": manifest["h2o_version"],
        "runtime_h2o_version": runtime_version,
        "mojo_version": manifest.get("mojo_version"),
    }
    prediction_rows = pd.DataFrame(
        {
            "row_id": valid_ids.astype(str),
            "prediction": predictions,
            **common,
            "aml_run_id": aml_run_id,
            "source_file": source_file,
            "scoring_time_utc": scoring_time,
            "correlation_id": args.correlation_id,
            "status": "scored",
        }
    )
    if len(prediction_rows) != len(valid):
        raise RuntimeError("Prediction count does not match valid input row count")
    monitoring = valid.copy()
    monitoring.insert(0, "row_id", valid_ids.astype(str))
    monitoring["prediction"] = predictions
    monitoring["scoring_time_utc"] = scoring_time
    monitoring["correlation_id"] = args.correlation_id
    statistics = pd.DataFrame({"feature": manifest["features"]})
    numeric = valid.select_dtypes(include=[np.number])
    if len(numeric) and len(numeric.columns):
        statistics = statistics.merge(
            numeric.describe().T.reset_index(names="feature"),
            on="feature",
            how="left",
        )
    statistics["invalid_count"] = statistics["feature"].map(invalid_counts).fillna(0).astype(int)
    prediction_rows.to_csv(scored_output / "predictions.csv", index=False)
    rejects.to_csv(scored_output / "rejects.csv", index=False)
    monitoring.to_csv(monitoring_output / "monitoring_data.csv", index=False)
    statistics.to_csv(monitoring_output / "feature_statistics.csv", index=False)
    duration = time.perf_counter() - started
    input_rows = len(valid) + len(rejects)
    numeric_predictions = pd.to_numeric(predictions, errors="coerce")
    predictions_are_numeric = len(predictions) > 0 and numeric_predictions.notna().all()
    summary = {
        **common,
        "aml_run_id": aml_run_id,
        "correlation_id": args.correlation_id,
        "input_rows": input_rows,
        "scored_rows": len(prediction_rows),
        "rejected_rows": len(rejects),
        "reject_rate": len(rejects) / input_rows,
        "duration_seconds": duration,
        "jvm_startup_seconds": jvm_seconds,
        "scoring_seconds": scoring_seconds,
        "rows_per_second": len(prediction_rows) / duration if duration else 0.0,
        "prediction_mean": float(numeric_predictions.mean()) if predictions_are_numeric else None,
        "prediction_std": float(numeric_predictions.std(ddof=0)) if predictions_are_numeric else None,
        "prediction_min": float(numeric_predictions.min()) if predictions_are_numeric else None,
        "prediction_max": float(numeric_predictions.max()) if predictions_are_numeric else None,
        "quality_gate": "passed",
    }
    run_manifest = {
        "schema_version": "1.0",
        "created_utc": scoring_time,
        "input": {"source_file": source_file, "rows": input_rows},
        "model": common,
        "execution": {
            "aml_run_id": aml_run_id,
            "correlation_id": args.correlation_id,
            "fail_on_rejects": args.fail_on_rejects,
        },
        "outputs": {
            "predictions": "predictions.csv",
            "rejects": "rejects.csv",
            "monitoring_data": "monitoring_data.csv",
            "feature_statistics": "feature_statistics.csv",
        },
    }
    (monitoring_output / "run_manifest.json").write_text(
        json.dumps(run_manifest, indent=2) + "\n", encoding="utf-8"
    )
    (monitoring_output / "summary.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    if os.getenv("AZUREML_RUN_ID"):
        mlflow.log_params({key: value for key, value in common.items() if value is not None})
        mlflow.log_metrics(
            {
                "input_rows": summary["input_rows"],
                "scored_rows": summary["scored_rows"],
                "rejected_rows": summary["rejected_rows"],
                "reject_rate": summary["reject_rate"],
                "duration_seconds": duration,
                "rows_per_second": summary["rows_per_second"],
            }
        )
        mlflow.log_artifact(str(monitoring_output / "feature_statistics.csv"), "monitoring")
        mlflow.log_artifact(str(monitoring_output / "run_manifest.json"), "monitoring")
    print(json.dumps(summary, sort_keys=True), flush=True)
    if args.fail_on_rejects and len(rejects):
        raise RuntimeError(f"Rejected {len(rejects)} rows")


if __name__ == "__main__":
    main()
'''

BATCH_SCORE_SOURCE = textwrap.dedent(BATCH_SCORE_SOURCE).lstrip()
ast.parse(BATCH_SCORE_SOURCE, filename=str(BATCH_SCORE_PATH))
BATCH_SCORE_PATH.write_text(BATCH_SCORE_SOURCE, encoding="utf-8")

COMPONENT_SOURCE = f'''$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: h2o_customer_score
display_name: Score Customer H2O Model
type: command
description: Score a native H2O binary or MOJO and emit monitoring outputs.
inputs:
  model_dir: {{type: custom_model}}
  input_data: {{type: uri_file}}
  correlation_id: {{type: string}}
  id_column: {{type: string, default: __generated__}}
  fail_on_rejects: {{type: boolean, default: false}}
  h2o_nthreads: {{type: integer, min: 1, default: 3}}
  h2o_max_mem_size: {{type: string, default: 6G}}
outputs:
  scored_output: {{type: uri_folder}}
  monitoring_output: {{type: uri_folder}}
code: ./code
environment: azureml:{ENVIRONMENT_NAME}@latest
command: >-
  python score.py --model-dir ${{{{inputs.model_dir}}}}
  --input-data ${{{{inputs.input_data}}}} --scored-output ${{{{outputs.scored_output}}}}
  --monitoring-output ${{{{outputs.monitoring_output}}}}
  --correlation-id '${{{{inputs.correlation_id}}}}' --id-column '${{{{inputs.id_column}}}}'
  --fail-on-rejects ${{{{inputs.fail_on_rejects}}}}
  --h2o-nthreads ${{{{inputs.h2o_nthreads}}}}
  --h2o-max-mem-size '${{{{inputs.h2o_max_mem_size}}}}'
'''
PIPELINE_SOURCE = f'''$schema: https://azuremlschemas.azureedge.net/latest/pipelineJob.schema.json
type: pipeline
experiment_name: {os.environ['H2O_CUSTOMER_EXPERIMENT_NAME']}
description: Notebook-generated customer H2O scoring pipeline.
inputs:
  model_dir: {{type: custom_model, path: 'azureml:{MODEL_NAME}@latest'}}
  input_data: {{type: uri_file, path: '{INPUT_PATH.as_posix()}'}}
  correlation_id: workshop-manual-run
  id_column: __generated__
  fail_on_rejects: false
  h2o_nthreads: 3
  h2o_max_mem_size: 6G
outputs:
  scored_output: {{mode: rw_mount}}
  monitoring_output: {{mode: rw_mount}}
settings:
  default_compute: azureml:{COMPUTE_NAME}
  default_datastore: azureml:{OUTPUT_DATASTORE}
  continue_on_step_failure: false
  force_rerun: true
jobs:
  score:
    type: command
    component: ./h2o-score.yaml
    inputs:
      model_dir: ${{{{parent.inputs.model_dir}}}}
      input_data: ${{{{parent.inputs.input_data}}}}
      correlation_id: ${{{{parent.inputs.correlation_id}}}}
      id_column: ${{{{parent.inputs.id_column}}}}
      fail_on_rejects: ${{{{parent.inputs.fail_on_rejects}}}}
      h2o_nthreads: ${{{{parent.inputs.h2o_nthreads}}}}
      h2o_max_mem_size: ${{{{parent.inputs.h2o_max_mem_size}}}}
    outputs:
      scored_output: ${{{{parent.outputs.scored_output}}}}
      monitoring_output: ${{{{parent.outputs.monitoring_output}}}}
'''
COMPONENT_SOURCE = textwrap.dedent(COMPONENT_SOURCE).lstrip()
PIPELINE_SOURCE = textwrap.dedent(PIPELINE_SOURCE).lstrip()
yaml.safe_load(COMPONENT_SOURCE)
yaml.safe_load(PIPELINE_SOURCE)
COMPONENT_PATH.write_text(COMPONENT_SOURCE, encoding="utf-8")
PIPELINE_PATH.write_text(PIPELINE_SOURCE, encoding="utf-8")
print(f"Generated scorer: {BATCH_SCORE_PATH}")
print(f"Generated component: {COMPONENT_PATH}")
print(f"Generated pipeline: {PIPELINE_PATH}")

## 5. Bind this run to the generated H2O pipeline

`load_job` reads the H2O YAML graph generated by the preceding cell. The notebook then supplies latest asset references, correlation ID, reject behavior, H2O limits, compute, and output datastore.

Review the printed summary as the run contract. Airflow should provide the same values when it submits this pipeline in production.

In [ ]:
pipeline_path = PIPELINE_PATH


def load_scoring_job(model_reference: str, input_reference: str):
    scoring_job = load_job(
        pipeline_path,
        params_override=[
            {"inputs.model_dir.path": model_reference},
            {"inputs.input_data.path": input_reference},
            {"inputs.correlation_id": CORRELATION_ID},
            {"inputs.id_column": ID_COLUMN},
            {"inputs.fail_on_rejects": FAIL_ON_REJECTS},
            {"inputs.h2o_nthreads": int(os.environ["H2O_NTHREADS"])},
            {"inputs.h2o_max_mem_size": os.environ["H2O_MAX_MEM_SIZE"]},
        ],
    )
    scoring_job.jobs["score"].component.environment = (
        f"azureml:{ENVIRONMENT_NAME}@latest"
    )
    scoring_job.settings.default_compute = COMPUTE_NAME
    scoring_job.settings.default_datastore = OUTPUT_DATASTORE
    scoring_job.identity = ManagedIdentityConfiguration(
        client_id=COMPUTE_IDENTITY_CLIENT_ID
    )
    for child_job in scoring_job.jobs.values():
        child_job.identity = ManagedIdentityConfiguration(
            client_id=COMPUTE_IDENTITY_CLIENT_ID
        )
    scoring_job.experiment_name = EXPERIMENT_NAME
    scoring_job.display_name = f"Customer H2O scoring - {CORRELATION_ID}"
    scoring_job.tags = {
        "workshop": "azureml-h2o-customer",
        "profile": manifest["profile"],
        "correlation_id": CORRELATION_ID,
        "model": model_reference,
    }
    scoring_job._validate(raise_error=True)
    return scoring_job


job = load_scoring_job(
    f"azureml:{MODEL_NAME}@latest",
    str(INPUT_PATH),
)

display(
    {
        "pipeline": str(pipeline_path),
        "display_name": job.display_name,
        "model": job.inputs["model_dir"].result(),
        "input": job.inputs["input_data"].result(),
        "environment": f"{ENVIRONMENT_NAME}@latest",
        "compute": job.settings.default_compute,
        "datastore": job.settings.default_datastore,
        "runtime_identity": COMPUTE_IDENTITY_CLIENT_ID,
        "fail_on_rejects": FAIL_ON_REJECTS,
    }
)

## 6. Register the input and submit

This is the only cell that writes to Azure. With submission enabled, it resolves the latest model and environment, registers the input file with an SDK-assigned version, submits the job, and streams its status.

The Studio URL is printed for operators who want to follow the run graph and component logs. After submission, return `RUN_H2O_SCORING_PIPELINE` to `false`.

In [ ]:
if RUN:
    registered_model = ml_client.models.get(MODEL_NAME, label="latest")
    registered_environment = ml_client.environments.get(
        ENVIRONMENT_NAME, label="latest"
    )
    model_reference = (
        f"azureml:{registered_model.name}:{registered_model.version}"
    )
    environment_reference = (
        f"azureml:{registered_environment.name}:{registered_environment.version}"
    )

    if GOLDEN_VALIDATION_REQUIRED:
        if not RUNTIME_VALIDATION_PATH.is_file():
            raise FileNotFoundError(
                "Required runtime validation evidence is missing; run notebook 04 first"
            )
        runtime_validation = json.loads(
            RUNTIME_VALIDATION_PATH.read_text(encoding="utf-8")
        )
        expected_validation = {
            "status": "passed",
            "model": f"{registered_model.name}:{registered_model.version}",
            "environment": (
                f"{registered_environment.name}:{registered_environment.version}"
            ),
        }
        for name, expected in expected_validation.items():
            if runtime_validation.get(name) != expected:
                raise RuntimeError(
                    f"Runtime validation {name} is {runtime_validation.get(name)!r}; "
                    f"expected {expected!r}"
                )

    registered_data = ml_client.data.create_or_update(data_definition)
    verified_data = ml_client.data.get(
        DATA_NAME, version=registered_data.version
    )
    data_reference = f"azureml:{verified_data.name}:{verified_data.version}"
    print(f"Input data: {verified_data.name}:{verified_data.version}")

    job = load_scoring_job(model_reference, data_reference)
    job.jobs["score"].component.environment = environment_reference
    job.tags["environment"] = environment_reference
    job._validate(raise_error=True)

    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted job: {submitted_job.name}")
    print(f"Studio URL: {submitted_job.studio_url}")

    ml_client.jobs.stream(submitted_job.name)
else:
    print(f"Prepared scoring pipeline for {MODEL_NAME}@latest")
    print("Submission is off. Set RUN_H2O_SCORING_PIPELINE=true when ready.")

## 7. Inspect the final outputs

After streaming finishes, we read the job again and require a `Completed` status. The returned output URIs are the handoff points for scored records and monitoring artifacts.

Airflow should capture the same job status and output references when it submits this pipeline in production.

In [18]:
if RUN:
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Pipeline ended with status {final_job.status}")

    output_paths = {
        name: output.path
        for name, output in final_job.outputs.items()
    }
    print(f"Pipeline status: {final_job.status}")
    display(output_paths)
else:
    print("No outputs to inspect because submission is off.")

Pipeline status: Completed


{'scored_output': None, 'monitoring_output': None}

## Expected Result

The customer input is registered with an Azure ML-assigned version, the notebook-generated command pipeline completes, and Airflow-ready scored and monitoring output URIs are returned.

Next: review `workshop/docs/AIRFLOW.md` and `workshop/docs/NEXT_STEPS.md`.